In [26]:
# ------------------------------------------------------------
# SQL Online Retail Store Database Project
# ------------------------------------------------------------
# This notebook generates a synthetic multi-table SQLite database
# to simulate a realistic online retail system.
#
# The project demonstrates:
# - Relational schema design
# - Foreign key relationships
# - Composite primary keys
# - Nominal, ordinal, interval and ratio data types
# - Deliberate insertion of missing and duplicate data
# - At least one table with more than 1000 rows (assignment requirement)
# ------------------------------------------------------------

In [2]:
# Install Faker library
# Faker is used to generate realistic synthetic data
# such as names, emails, dates, and review text.
# This ensures the dataset does not contain real personal data
# and supports ethical data generation.
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.0 MB/s eta 0:00:00


In [4]:
# ------------------------------------------------------------
# Import Required Libraries
# ------------------------------------------------------------

# sqlite3: Used to create and manage the SQLite relational database
import sqlite3

# random: Used for random selections such as categories, order status,
# and simulating variability in transactional behaviour
import random

# Faker: Used to generate realistic synthetic data (names, emails,
# dates, review text) without using real personal information
from faker import Faker

# numpy: Used to generate realistic statistical distributions,
# such as normally distributed annual income values
import numpy as np

In [5]:
# ------------------------------------------------------------
# Create SQLite Database Connection
# ------------------------------------------------------------

# Create a connection to a SQLite database file named 'retail_store.db'.
# SQLite is chosen because it is lightweight, file-based,
# and suitable for demonstrating relational database design.
conn = sqlite3.connect("retail_store.db")

# Create a cursor object to execute SQL commands
# such as CREATE TABLE, INSERT, UPDATE, and SELECT statements.
cursor = conn.cursor()

In [6]:
# ------------------------------------------------------------
# Create Customers Table
# ------------------------------------------------------------
# This table stores demographic and membership information
# for each customer in the retail system.
#
# Data types included:
# - Nominal data: gender
# - Ordinal data: membership_level (Bronze < Silver < Gold < Platinum)
# - Interval data: date_of_birth
# - Ratio data: annual_income (true zero and meaningful comparison)
#
# The membership_level column includes a CHECK constraint
# to ensure only valid ordered categories are inserted.
# The customer_id is the primary key to uniquely identify each customer.
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE Customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    gender TEXT,
    membership_level TEXT CHECK(membership_level IN ('Bronze','Silver','Gold','Platinum')),
    date_of_birth DATE,
    email TEXT,
    annual_income REAL
);
""")

In [7]:
# ------------------------------------------------------------
# Create Products Table
# ------------------------------------------------------------
# This table stores product inventory information.
#
# Data types included:
# - Nominal data: category
# - Ratio data: price, stock_quantity, weight_grams
# - supplier_rating is stored as a numerical scale (1–5)
#
# The price column includes a CHECK constraint (price > 0)
# to ensure logical data validity.
#
# product_id is defined as the primary key to uniquely
# identify each product in the system.
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE Products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    price REAL CHECK(price > 0),
    stock_quantity INTEGER,
    supplier_rating REAL,
    weight_grams REAL
);
""")

In [8]:
# ------------------------------------------------------------
# Create Orders Table
# ------------------------------------------------------------
# This table represents customer purchase transactions.
#
# Data types included:
# - Interval data: order_date
# - Nominal data: shipping_method
# - Ordinal data: order_status
#   (Pending < Shipped < Delivered < Cancelled)
# - Ratio data: total_amount
#
# customer_id is defined as a foreign key referencing
# Customers(customer_id). This enforces referential integrity
# and ensures that every order is linked to a valid customer.
#
# order_status includes a CHECK constraint to ensure only
# valid ordered categories are inserted.
#
# order_id is the primary key to uniquely identify each order.
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE,
    shipping_method TEXT,
    order_status TEXT CHECK(order_status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
);
""")

In [9]:
# ------------------------------------------------------------
# Create Order_Items Table
# ------------------------------------------------------------
# This table resolves the many-to-many relationship
# between Orders and Products.
#
# In a retail system:
# - One order can contain multiple products
# - One product can appear in multiple orders
#
# To correctly model this relationship, a separate junction
# table (Order_Items) is created.
#
# The composite primary key (order_id, product_id)
# ensures that the same product cannot be inserted
# more than once within the same order.
#
# Foreign keys enforce referential integrity:
# - order_id references Orders(order_id)
# - product_id references Products(product_id)
#
# quantity includes a CHECK constraint to ensure
# only positive values are allowed.
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE Order_Items (
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER CHECK(quantity > 0),
    unit_price REAL,
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES Orders(order_id),
    FOREIGN KEY (product_id) REFERENCES Products(product_id)
);
""")

In [10]:
# ------------------------------------------------------------
# Create Payments Table
# ------------------------------------------------------------
# This table stores payment information for each order.
#
# Relationship:
# - Each order has one corresponding payment record.
# - This represents a one-to-one relationship between
#   Orders and Payments.
#
# Data types included:
# - Nominal data: payment_method
# - Interval data: payment_date
# - Ratio data: amount_paid
#
# order_id is defined as a foreign key referencing Orders(order_id)
# to enforce referential integrity and ensure that
# every payment corresponds to a valid order.
#
# payment_id is the primary key to uniquely identify
# each payment transaction.
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE Payments (
    payment_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    payment_method TEXT,
    payment_date DATE,
    amount_paid REAL,
    payment_status TEXT,
    FOREIGN KEY (order_id) REFERENCES Orders(order_id)
);
""")

In [11]:
# ------------------------------------------------------------
# Create Reviews Table
# ------------------------------------------------------------
# This table stores customer feedback on purchased products.
#
# Relationship:
# - One customer can review multiple products.
# - One product can receive reviews from multiple customers.
# This creates a many-to-many relationship between
# Customers and Products, resolved through this table.
#
# Data types included:
# - Ordinal data: rating (1–5 scale)
# - Interval data: review_date
# - Nominal/Text data: review_text
#
# The rating column includes a CHECK constraint
# to ensure values are between 1 and 5 only.
#
# Foreign keys enforce referential integrity:
# - product_id references Products(product_id)
# - customer_id references Customers(customer_id)
#
# review_id is the primary key to uniquely identify each review.
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE Reviews (
    review_id INTEGER PRIMARY KEY,
    product_id INTEGER,
    customer_id INTEGER,
    rating INTEGER CHECK(rating BETWEEN 1 AND 5),
    review_text TEXT,
    review_date DATE,
    FOREIGN KEY (product_id) REFERENCES Products(product_id),
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
);
""")

In [12]:
# ------------------------------------------------------------
# Commit Changes to the Database
# ------------------------------------------------------------
# The commit() function permanently saves all executed
# SQL statements (CREATE TABLE, INSERT, UPDATE, etc.)
# to the SQLite database file.
#
# Without committing, changes would only exist in memory
# and would be lost once the connection is closed.
# ------------------------------------------------------------

conn.commit()

In [13]:
# ------------------------------------------------------------
# Generate Synthetic Customer Data
# ------------------------------------------------------------
# Faker is used to generate realistic synthetic data
# such as names, email addresses, and dates of birth.
# This ensures that no real personal data is used.
#
# membership_levels represent ordinal categories
# (Bronze < Silver < Gold < Platinum).
#
# Income is generated using a normal distribution
# (mean=35000, std=10000) to simulate realistic
# income variation in a retail customer base.
#
# 5% of annual_income values are deliberately set to NULL
# to simulate missing data commonly found in real-world datasets.
# ------------------------------------------------------------

fake = Faker()

membership_levels = ["Bronze", "Silver", "Gold", "Platinum"]

for i in range(1, 501):

    # Generate income using normal distribution
    income = round(np.random.normal(35000, 10000), 2)

    # Introduce 5% missing income values (realistic data imperfection)
    if random.random() < 0.05:
        income = None

    cursor.execute("""
    INSERT INTO Customers VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        i,
        fake.first_name(),
        fake.last_name(),
        random.choice(["Male", "Female"]),  # Nominal category
        random.choice(membership_levels),   # Ordinal category
        fake.date_of_birth(minimum_age=18, maximum_age=70),
        fake.email(),
        income
    ))

# Save inserted records permanently
conn.commit()

/tmp/ipykernel_195/1906171096.py:32: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [14]:
# ------------------------------------------------------------
# Generate Synthetic Product Data
# ------------------------------------------------------------
# Product categories represent nominal categorical data.
# Prices are generated using a uniform distribution between
# £5 and £500 to simulate a realistic retail price range.
#
# stock_quantity is generated between 0 and 1000 to reflect
# real inventory variation (including out-of-stock products).
#
# supplier_rating is generated between 1 and 5 (1 decimal place),
# representing a continuous rating scale.
#
# weight_grams simulates product weight variation between
# small and heavy products (100g to 5000g).
#
# Randomisation ensures realistic variability in the dataset.
# ------------------------------------------------------------

categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]

for i in range(1, 201):

    cursor.execute("""
    INSERT INTO Products VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        i,
        fake.word().capitalize(),           # Product name
        random.choice(categories),          # Nominal category
        round(random.uniform(5, 500), 2),   # Ratio data (price)
        random.randint(0, 1000),            # Inventory level
        round(random.uniform(1, 5), 1),     # Supplier rating
        round(random.uniform(100, 5000), 2) # Weight in grams
    ))

# Commit product records to database
conn.commit()

In [15]:
# ------------------------------------------------------------
# Generate Synthetic Order Data
# ------------------------------------------------------------
# This section creates 1500 customer orders.
#
# customer_id is randomly assigned between 1 and 500,
# simulating repeat purchases by customers.
#
# order_date is restricted to the current year to
# represent a realistic retail dataset timeframe.
#
# shipping_method is nominal categorical data.
#
# order_status represents ordinal stages in the
# order lifecycle (Pending < Shipped < Delivered < Cancelled).
#
# total_amount is generated between £10 and £1000,
# reflecting small to large retail purchases.
#
# Randomisation ensures realistic purchasing behaviour,
# including multiple orders per customer.
# ------------------------------------------------------------

order_status = ["Pending", "Shipped", "Delivered", "Cancelled"]

for i in range(1, 1501):

    cursor.execute("""
    INSERT INTO Orders VALUES (?, ?, ?, ?, ?, ?)
    """, (
        i,
        random.randint(1, 500),               # Many-to-one relationship (Customers)
        fake.date_this_year(),                # Interval data
        random.choice(["Standard", "Express"]),
        random.choice(order_status),          # Ordinal category
        round(random.uniform(10, 1000), 2)    # Ratio data
    ))

# Save all order records
conn.commit()

/tmp/ipykernel_195/3659164402.py:28: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [16]:
# ------------------------------------------------------------
# Validate Inserted Record Counts
# ------------------------------------------------------------
# This section verifies that the expected number of
# records has been successfully inserted into each table.
#
# Performing validation checks is good database practice
# to ensure data generation and insertion completed correctly.
# ------------------------------------------------------------

cursor.execute("SELECT COUNT(*) FROM Customers")
print("Customers:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM Products")
print("Products:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM Orders")
print("Orders:", cursor.fetchone()[0])

Customers: 500
Products: 200
Orders: 1500


In [17]:
# ------------------------------------------------------------
# Generate Order_Items (Junction Table Data)
# ------------------------------------------------------------
# This section populates the Order_Items table, which
# resolves the many-to-many relationship between
# Orders and Products.
#
# Each order contains between 1 and 5 products,
# reflecting realistic retail purchasing behaviour.
#
# random.sample() ensures that the same product
# cannot appear more than once within a single order,
# which aligns with the composite primary key
# (order_id, product_id).
#
# quantity is restricted between 1 and 5 units,
# simulating realistic basket quantities.
#
# unit_price is generated independently to simulate
# price variation (e.g., discounts or promotions).
# ------------------------------------------------------------

for order_id in range(1, 1501):

    # Each order will have 1 to 5 unique products
    number_of_items = random.randint(1, 5)

    # Select unique product IDs for this order
    products_in_order = random.sample(range(1, 201), number_of_items)

    for product_id in products_in_order:

        quantity = random.randint(1, 5)
        unit_price = round(random.uniform(5, 500), 2)

        cursor.execute("""
        INSERT INTO Order_Items VALUES (?, ?, ?, ?)
        """, (
            order_id,
            product_id,
            quantity,
            unit_price
        ))

# Commit all order item records
conn.commit()

In [18]:
# ------------------------------------------------------------
# Validate Order_Items Record Count
# ------------------------------------------------------------
# This query verifies the total number of records inserted
# into the Order_Items junction table.
#
# Because each order contains between 1 and 5 products,
# the total number of Order_Items records should be
# significantly higher than the number of Orders.
#
# This confirms that the many-to-many relationship
# has been populated correctly.
# ------------------------------------------------------------

cursor.execute("SELECT COUNT(*) FROM Order_Items")
print("Order_Items:", cursor.fetchone()[0])

Order_Items: 4435


In [19]:
# ------------------------------------------------------------
# Generate Payment Data
# ------------------------------------------------------------
# This section creates one payment record per order,
# modelling a one-to-one relationship between
# Orders and Payments.
#
# payment_id and order_id are aligned intentionally,
# ensuring each order has exactly one payment entry.
#
# payment_method represents nominal categorical data.
#
# payment_status reflects transaction outcomes.
# In real-world systems, most payments are completed,
# with a small proportion failing or refunded.
#
# amount_paid is generated within the same range as
# order total values to simulate realistic transaction sizes.
# ------------------------------------------------------------

payment_methods = ["Credit Card", "Debit Card", "PayPal", "Bank Transfer"]
payment_status = ["Completed", "Failed", "Refunded"]

for i in range(1, 1501):

    cursor.execute("""
    INSERT INTO Payments VALUES (?, ?, ?, ?, ?, ?)
    """, (
        i,
        i,  # One-to-one mapping with Orders
        random.choice(payment_methods),
        fake.date_this_year(),  # Interval data
        round(random.uniform(10, 1000), 2),
        random.choices(
            payment_status,
            weights=[0.85, 0.1, 0.05],  # More realistic distribution
            k=1
        )[0]
    ))

# Commit payment records
conn.commit()

/tmp/ipykernel_195/3092813094.py:26: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [20]:
# ------------------------------------------------------------
# Validate Payments Record Count
# ------------------------------------------------------------
# This query confirms that a payment record has been
# successfully created for each order.
#
# Since the design models a one-to-one relationship
# between Orders and Payments, the total number of
# payment records should match the number of orders.
#
# This validation step ensures referential integrity
# and confirms correct data generation.
# ------------------------------------------------------------

cursor.execute("SELECT COUNT(*) FROM Payments")
print("Payments:", cursor.fetchone()[0])

Payments: 1500


In [21]:
# ------------------------------------------------------------
# Generate Customer Review Data
# ------------------------------------------------------------
# This section creates 800 synthetic product reviews.
#
# The Reviews table models a many-to-many relationship
# between Customers and Products:
# - One customer can review multiple products.
# - One product can receive multiple reviews.
#
# rating represents ordinal data on a 1–5 scale.
# review_text is generated using Faker to simulate
# realistic written feedback.
#
# Only a subset of orders typically result in reviews,
# therefore 800 reviews (less than total orders) reflects
# realistic customer engagement behaviour.
#
# Random selection of product_id and customer_id
# simulates natural variation in reviewing patterns.
# ------------------------------------------------------------

review_id = 1

for _ in range(800):  # 800 reviews (not every order generates a review)

    cursor.execute("""
    INSERT INTO Reviews VALUES (?, ?, ?, ?, ?, ?)
    """, (
        review_id,
        random.randint(1, 200),      # product_id
        random.randint(1, 500),      # customer_id
        random.randint(1, 5),        # Ordinal rating (1–5)
        fake.sentence(),             # Synthetic review text
        fake.date_this_year()        # Interval data
    ))

    review_id += 1

# Commit review records to database
conn.commit()

/tmp/ipykernel_195/1484126560.py:27: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""


In [22]:
# ------------------------------------------------------------
# Validate Reviews Record Count
# ------------------------------------------------------------
# This query confirms the total number of review records
# successfully inserted into the Reviews table.
#
# Since not all customers leave reviews, the total number
# of reviews is intentionally lower than the number of orders.
# This reflects realistic post-purchase engagement behaviour.
# ------------------------------------------------------------

cursor.execute("SELECT COUNT(*) FROM Reviews")
print("Reviews:", cursor.fetchone()[0])

Reviews: 800


In [23]:
# ------------------------------------------------------------
# Introduce Deliberate Duplicate Emails (Data Realism)
# ------------------------------------------------------------
# Real-world datasets often contain duplicate entries
# due to user input errors, shared email accounts,
# or system inconsistencies.
#
# To simulate this realistically, 10 existing emails
# are selected and reassigned to 20 random customers.
# This introduces approximately 2–4% duplicate values
# in the email column.
#
# This controlled imperfection improves realism and
# demonstrates understanding of data quality issues.
# ------------------------------------------------------------

# Get 10 existing customer emails
cursor.execute("SELECT email FROM Customers LIMIT 10")
duplicate_emails = cursor.fetchall()

# Reassign those emails to random other customers
for _ in range(20):
    cursor.execute("""
    UPDATE Customers
    SET email = ?
    WHERE customer_id = ?
    """, (
        duplicate_emails[random.randint(0, 9)][0],
        random.randint(1, 500)
    ))

# Commit duplicate changes
conn.commit()

In [24]:
# ------------------------------------------------------------
# Introduce Missing Review Text (Data Imperfection Simulation)
# ------------------------------------------------------------
# In real-world systems, some customers submit ratings
# without written feedback, or text may be lost due to
# system errors.
#
# To simulate this realistic data imperfection,
# 50 review_text values (~6% of reviews) are deliberately
# set to NULL.
#
# This demonstrates handling of incomplete textual data
# and improves dataset authenticity.
# ------------------------------------------------------------

for _ in range(50):  # Approximately 6% missing review text
    cursor.execute("""
    UPDATE Reviews
    SET review_text = NULL
    WHERE review_id = ?
    """, (random.randint(1, 800),))

conn.commit()

In [25]:
# ------------------------------------------------------------
# Close Database Connection
# ------------------------------------------------------------
# The close() function safely terminates the connection
# to the SQLite database.
#
# Closing the connection ensures:
# - All transactions are finalised
# - System resources are released
# - No further accidental modifications occur
#
# This represents good database management practice.
# ------------------------------------------------------------

conn.close()